# Malware Detection Model

## Purpose
This notebook builds a binary classifier that predicts whether a Win32
executable is **malicious or benign**, using a **LightGBM** (`LGBMClassifier`)
model trained on engineered features derived from static file metadata.

## Data
Two pre-split parquet files are loaded:
- `win32_detection_train_20pct.parquet` — a 20% sample of labeled training data
- `win32_test_detection.parquet` — a held-out labeled test set

Each row represents one Win32 file and includes:
- `imports` — JSON describing which DLLs and functions the file imports
- `strings` — JSON with summary statistics of printable strings found in the file
- `general` — JSON with basic file metadata (size, entropy, PE validity)
- `label` — ground-truth target (malicious vs. benign)

## Feature Engineering
The raw `imports`, `strings`, and `general` columns are JSON blobs, so three
helper functions convert them into flat numeric feature columns that a
tree-based model can consume:

| Function | Source column | Output features |
|---|---|---|
| `extract_import_features` | `imports` | # imported DLLs, # imported functions, avg functions/DLL, presence flags for 6 common system DLLs (`KERNEL32.dll`, `ADVAPI32.dll`, `USER32.dll`, `WS2_32.dll`, `SHELL32.dll`, `OLE32.dll`) |
| `extract_string_features` | `strings` | string count, average string length, printable character count, string entropy |
| `extract_general_features` | `general` | file size, file entropy, is-PE flag |

Each function is applied row-wise (`.apply(...)`) to a column, producing a
Series of dicts that is expanded into its own feature `DataFrame`. These are
then concatenated column-wise to form the full feature matrix:
- `X_train` = `general_features` + `strings_features` + `imports_features` (from `train_df`)
- `X_test` = the same three blocks computed from `test_df`

`y_train` / `y_test` come directly from the `label` column in each dataset.

## Modeling
An `LGBMClassifier` is trained with:
- `n_estimators=100`
- `learning_rate=0.1`
- `random_state=42`

fit on `X_train` / `y_train`.

## Evaluation
The trained model predicts on `X_test`, producing both hard class labels and
malicious-class probabilities. Performance is reported using:
- **Accuracy**
- **ROC AUC**
- **Confusion matrix**
- **Classification report** (precision, recall, F1 per class)

## Notebook Flow (cell-by-cell)
1. **Imports** — load pandas, LightGBM, and sklearn metrics.
2. **Load data** — read train/test parquet files, inspect shape/schema.
3. **Feature engineering** – function definitions, applied to train and test sets.
4. Concatenate the three feature blocks → `X_train` and `X_test`, extract `y_train` and `y_test`.
5. Instantiate and fit the `LGBMClassifier` → `model`.
6. Predict on `X_test`.
7. Print accuracy, AUC, confusion matrix, and classification report.

In [22]:
# imports
from pathlib import Path
import json
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

In [23]:
ROOT = Path.cwd().parent
DATA_DIR = ROOT / "win32_data"
train_df = pd.read_parquet(DATA_DIR / "win32_detection_train_20pct.parquet")
test_df = pd.read_parquet(DATA_DIR / "win32_test_detection.parquet")

print("Train preview")
display(train_df.head())
print("Test preview")
display(test_df.head())

print("Train info")
display(train_df.info())
print("Test info")
display(test_df.info())

# preview to values in imports  
# print(train_df["imports"].iloc[0][:500])

Train preview


,sha256,label,general,strings,imports
0,000742aa22166e00603ab1101adfd75c793f6ff38cc3f9...,0,"{""size"": 4579380, ""entropy"": 7.98444658339403,...","{""numstrings"": 22744, ""avlength"": 5.8437829757...","{""KERNEL32.dll"": [""GetDriveTypeA"", ""GetModuleF..."
1,000adea549ab7604629f3b606bbcf2afd5aec6f7ed0e6c...,0,"{""size"": 152576, ""entropy"": 3.710459087578284,...","{""numstrings"": 19, ""avlength"": 24.473684210526...",{}
2,0015cbb011bbeb891b3ea505a6995d17395c4d4fd144b0...,1,"{""size"": 122880, ""entropy"": 7.2352091959923825...","{""numstrings"": 838, ""avlength"": 33.65632458233...","{""wsock32.dll"": [""WSAGetLastError"", ""WSAStartu..."
3,001ab6c768246a86b9c784630fda6574980980da5e11bd...,0,"{""size"": 34392, ""entropy"": 6.492246788417963, ...","{""numstrings"": 291, ""avlength"": 14.57731958762...","{""ADVAPI32.dll"": [""RegCloseKey"", ""RegCreateKey..."
4,0020532d7497a685cf1d75668eef3184a0e683fd620090...,1,"{""size"": 437224, ""entropy"": 6.436412852928497,...","{""numstrings"": 9960, ""avlength"": 10.5661646586...","{""MFC42.DLL"": [""MFC42.DLL:ordinal2770"", ""MFC42..."


Test preview


,sha256,label,general,strings,imports
0,00005da006489e8c288487dc2899c8afe341a610887ae7...,0,"{""size"": 123848, ""entropy"": 5.200856607961974,...","{""numstrings"": 912, ""avlength"": 11.49561403508...",{}
1,00014360be92b703d45d6db1c09f0916cdc580720fedc4...,1,"{""size"": 3000000, ""entropy"": 7.06598938246347,...","{""numstrings"": 22731, ""avlength"": 31.684175795...","{""oleaut32.dll"": [""SafeArrayPtrOfIndex"", ""Safe..."
2,00017862cd0c57bbabb30a3698f0181cdc5bb93c2c94db...,0,"{""size"": 884888, ""entropy"": 6.4442233179278325...","{""numstrings"": 5972, ""avlength"": 22.1989283322...","{""WININET.dll"": [""HttpSendRequestA"", ""HttpQuer..."
3,0001b778405e30b1bc7482907e479bc5eee255777c5c7e...,1,"{""size"": 135168, ""entropy"": 7.326132781964504,...","{""numstrings"": 632, ""avlength"": 7.704113924050...","{""MSVBVM60.DLL"": [""__vbaVarTstGt"", ""__vbaVarSu..."
4,00037ce21a123ae05be26f44b07fd45852c7100b70d6ec...,1,"{""size"": 238080, ""entropy"": 7.361566089709001,...","{""numstrings"": 1064, ""avlength"": 7.64567669172...","{""kernel32.dll"": [""LoadLibraryA"", ""GetProcAddr..."


Train info
<class 'pandas.DataFrame'>
RangeIndex: 312125 entries, 0 to 312124
Data columns (total 5 columns):
 #   Column   Non-Null Count   Dtype
---  ------   --------------   -----
 0   sha256   312125 non-null  str  
 1   label    312125 non-null  int64
 2   general  312125 non-null  str  
 3   strings  312125 non-null  str  
 4   imports  312125 non-null  str  
dtypes: int64(1), str(4)
memory usage: 1.5 GB


None

Test info
<class 'pandas.DataFrame'>
RangeIndex: 359994 entries, 0 to 359993
Data columns (total 5 columns):
 #   Column   Non-Null Count   Dtype
---  ------   --------------   -----
 0   sha256   359994 non-null  str  
 1   label    359994 non-null  int64
 2   general  359994 non-null  str  
 3   strings  359994 non-null  str  
 4   imports  359994 non-null  str  
dtypes: int64(1), str(4)
memory usage: 1.5 GB


None

### Feature Engineering Functions

In [ ]:
def extract_import_features(x):
    """
    Convert imports JSON into compact numerical features.
    """
    if isinstance(x, str):
        x = json.loads(x)

    if not isinstance(x, dict):
        return {}

    features = {}

    # number of imported DLLs
    features["num_imported_dlls"] = len(x)

    # total imported functions
    features["num_imported_functions"] = sum(
        len(funcs) for funcs in x.values()
    )

    # Average functions per DLL
    if len(x) > 0:
        features["avg_functions_per_dll"] = (
            features["num_imported_functions"] / len(x)
        )
    else:
        features["avg_functions_per_dll"] = 0

    # DLL presence indicators
    common_dlls = [
        "KERNEL32.dll",
        "ADVAPI32.dll",
        "USER32.dll",
        "WS2_32.dll",
        "SHELL32.dll",
        "OLE32.dll"
    ]

    for dll in common_dlls:
        features[f"has_{dll}"] = int(dll in x)

    return features

In [ ]:
def extract_string_features(x):
    if isinstance(x, str):
        x = json.loads(x)

    return {
        "numstrings": x.get("numstrings", 0),
        "avg_string_length": x.get("avlength", 0),
        "num_printables": x.get("printables", 0),
        "string_entropy": x.get("entropy", 0),
    }

In [ ]:
def extract_general_features(x):
    if isinstance(x, str):
        x = json.loads(x)

    return {
        "file_size": x.get("size", 0),
        "file_entropy": x.get("entropy", 0),
        "is_pe": x.get("is_pe", 0),
    }

### Vectorize Train and Test Dataframes

In [17]:
# train set vectorization 
imports_features = train_df["imports"].apply(extract_import_features)
imports_features = pd.DataFrame(imports_features.tolist())

strings_features = train_df["strings"].apply(extract_string_features)
strings_features = pd.DataFrame(strings_features.tolist())

general_features = train_df["general"].apply(extract_general_features)
general_features = pd.DataFrame(general_features.tolist())

# check values
print(f"Import features shape: {imports_features.shape}")
print("Import features preview:")
display(imports_features.head())


print(f"String features shape: {strings_features.shape}")
print("String features preview:")
display(strings_features.head())

print(f"General features shape: {general_features.shape}")
print("General features preview:")
display(general_features.head())

Import features shape: (312125, 9)
Import features preview:


,num_imported_dlls,num_imported_functions,avg_functions_per_dll,has_KERNEL32.dll,has_ADVAPI32.dll,has_USER32.dll,has_WS2_32.dll,has_SHELL32.dll,has_OLE32.dll
0,9,214,23.777778,1,1,1,0,1,0
1,0,0,0.000000,0,0,0,0,0,0
2,12,185,15.416667,0,0,0,0,0,0
3,6,53,8.833333,1,1,0,0,1,0
4,5,72,14.400000,1,0,1,0,1,0


String features shape: (312125, 4)
String features preview:


,numstrings,avg_string_length,num_printables,string_entropy
0,22744,5.843783,132911,6.573110
1,19,24.473684,465,4.989281
2,838,33.656325,28204,4.894263
3,291,14.577320,4242,5.685436
4,9960,10.566165,105239,4.000204


General features shape: (312125, 3)
General features preview:


,file_size,file_entropy,is_pe
0,4579380,7.984447,1
1,152576,3.710459,1
2,122880,7.235209,1
3,34392,6.492247,1
4,437224,6.436413,1


In [18]:
# test dataset vectorization
test_imports_features = test_df["imports"].apply(extract_import_features)
test_imports_features = pd.DataFrame(test_imports_features.tolist())

test_strings_features = test_df["strings"].apply(extract_string_features)
test_strings_features = pd.DataFrame(test_strings_features.tolist())

test_general_features = test_df["general"].apply(extract_general_features)
test_general_features = pd.DataFrame(test_general_features.tolist())

# check values
print(f"(test) Import features shape: {test_imports_features.shape}")
print("(test) Import features preview:")
display(test_imports_features.head())


print(f"(test) String features shape: {test_strings_features.shape}")
print("(test) String features preview:")
display(test_strings_features.head())

print(f"(test) General features shape: {test_general_features.shape}")
print("(test) General features preview:")
display(test_general_features.head())

(test) Import features shape: (359994, 9)
(test) Import features preview:


,num_imported_dlls,num_imported_functions,avg_functions_per_dll,has_KERNEL32.dll,has_ADVAPI32.dll,has_USER32.dll,has_WS2_32.dll,has_SHELL32.dll,has_OLE32.dll
0,0,0,0.000000,0,0,0,0,0,0
1,13,334,25.692308,0,0,0,0,0,0
2,18,293,16.277778,1,1,1,0,0,0
3,1,88,88.000000,0,0,0,0,0,0
4,1,4,4.000000,0,0,0,0,0,0


(test) String features shape: (359994, 4)
(test) String features preview:


,numstrings,avg_string_length,num_printables,string_entropy
0,912,11.495614,10484,2.904376
1,22731,31.684176,720213,5.104248
2,5972,22.198928,132572,5.688327
3,632,7.704114,4869,6.206506
4,1064,7.645677,8135,6.371429


(test) General features shape: (359994, 3)
(test) General features preview:


,file_size,file_entropy,is_pe
0,123848,5.200857,1
1,3000000,7.065989,1
2,884888,6.444223,1
3,135168,7.326133,1
4,238080,7.361566,1


### Build X_train, y_train, X_test, and y_test

In [19]:
# combine vectorized features for train 
X_train = pd.concat(
    [
        general_features,
        strings_features,
        imports_features
    ],
    axis=1
)
y_train = train_df["label"]

# combine vectorized features for train 
X_test = pd.concat(
    [
        test_general_features,
        test_strings_features,
        test_imports_features
    ],
    axis=1
)
y_test = test_df["label"]

# print shape
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

X_train shape: (312125, 16)
X_test shape: (359994, 16)


### Training/Testing Model

In [10]:
model = LGBMClassifier(
    n_estimators=100,
    learning_rate=0.1,
    random_state=42
)

model.fit(X_train, y_train)

[LightGBM] [Info] Number of positive: 156357, number of negative: 155768
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.005725 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2143
[LightGBM] [Info] Number of data points in the train set: 312125, number of used features: 16
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500944 -> initscore=0.003774
[LightGBM] [Info] Start training from score 0.003774


,random_state,42
,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.1
,n_estimators,100
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001


In [20]:
# predict class labels for the test set
y_pred = model.predict(X_test)

# predict probability of the positive class (1) for the test set
y_prob = model.predict_proba(X_test)[:, 1]

### Evaluation

In [21]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print("AUC:", roc_auc_score(y_test, y_prob))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.8792035422812603
AUC: 0.9574256706242059

Confusion Matrix:
[[165638  14356]
 [ 29130 150870]]

Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.92      0.88    179994
           1       0.91      0.84      0.87    180000

    accuracy                           0.88    359994
   macro avg       0.88      0.88      0.88    359994
weighted avg       0.88      0.88      0.88    359994

